## Intro

with HuggingFace's Serverless Inference API, a world of AI super power is now made possible at zero-cost. This post focus on the power of Automatic Speech Recognition (ASR). 

Previously we have talked about [using OpenAI's Whisper](openai-whisper.qmd). But that requires a pip install and local inference.

Here we'll show that SOTA ASR is accessible with just an API call!

First, we'll define some API calling functions.

In [8]:
#| output: false
#| echo: false
#| code-fold: false
# load our HF API Token
from dotenv import load_dotenv
load_dotenv('./secrets.env')

True

In [1]:
#| code-fold: false
import requests, os, base64, json
from IPython.display import Audio
from io import BytesIO

def data_query(filename: str, api_url: str):
    ''' sending data as binary bytes'''
    assert os.environ.get('HUGGINGFACEHUB_API_TOKEN'), f'HUGGINGFACEHUB_API_TOKEN not set in env'
    headers = {"Authorization": f"Bearer {os.environ['HUGGINGFACEHUB_API_TOKEN']}"}    

    with open(filename, "rb") as f:
        data = f.read()
    response = requests.post(api_url, headers=headers, data=data)
    return response.json()

def json_query(payload, api_url: str):
    '''sending more complex payload with parameters'''
    assert os.environ.get('HUGGINGFACEHUB_API_TOKEN'), f'HUGGINGFACEHUB_API_TOKEN not set in env'
    headers = {"Authorization": f"Bearer {os.environ['HUGGINGFACEHUB_API_TOKEN']}"}

    response = requests.post(api_url, headers=headers, json=payload)
    return response.content

def base64_encode_audio(filename: str, get_dataUrl: bool = False):
    fname, fext = os.path.splitext(os.path.basename(filename))
    with open(filename, "rb") as f:
        audio_data = f.read()
    audio_str = base64.b64encode(audio_data).decode('utf-8')
    # example: f'data:audio/octet-stream;base64,{audio_str}'
    return f'data:audio/{fext[1:]};base64,{audio_str}' if get_dataUrl else audio_str

## Simple Transcript Generation
From [this list of available models](https://huggingface.co/models?pipeline_tag=automatic-speech-recognition&sort=trending) I chose [OpenAI's Whisper Large v3 Turbo](https://huggingface.co/openai/whisper-large-v3-turbo)

And for this example, we are going to recycle the audio from [this previous Whisper vs Youtube Transcript API show-down](whisper_vs_ytranscript.qmd):

In [2]:
#| code-fold: true
audio_b64_data = base64_encode_audio("DeepSeek.m4a", get_dataUrl= False)
sound_bytes = base64.b64decode(audio_b64_data)
Audio(sound_bytes)

To generate just the transcript, we only need to send the audio in binary format to the model's endpoint:

In [10]:
#| code-fold: false
%%time
asr_model_url = "https://api-inference.huggingface.co/models/openai/whisper-large-v3-turbo"
output_asr = data_query("DeepSeek.m4a", api_url= asr_model_url)
print(output_asr['text'])

 As big tech is getting hammered in today's selloff, CNBC's Magnificent 7 index dropping more than 1%. Well, there's a new emerging threat to mega caps, massive spending in America's dominance in the AI race. Deirdre Bosa digs into that for today's tech check. Hey, Dee. Hey, good morning, Leslie. So here's a name that our audience may want to write down, DeepSeq. This is a new free open source AI model that beats the latest open AI and meta models on key benchmarks. And it was made for a fraction of a fraction of the cost. Now, it was trained by a Chinese research lab that used NVIDIA H800s. That's a lower performance version of the H100 chips that are cheaper, more available and tailored for restricted markets like China. Now, I've been testing it out this morning and on the surface, it looks and acts just like open AI's chat GPT. And in fact, it actually thinks that is chat GPT. When I asked what model are you, it answered, I'm an AI language model created by open AI specifically bas

## Transcript with Timestamp
if we want more granular details for downstream task, perhaps we might like want the timestamp.

In that case, we need to send a JSON payload with multiple parameters (see [the official doc](https://huggingface.co/docs/api-inference/tasks/automatic-speech-recognition) for a full list of configurable params):

In [7]:
#| code-fold: false
%%time
payload = {
    'inputs': audio_b64_data,
    'parameters': {'return_timestamps': True}
}
output_asr = json_query(payload, api_url= asr_model_url)
output_asr = json.loads(output_asr.decode('utf-8'))
output_asr['chunks']

CPU times: user 44.1 ms, sys: 34.5 ms, total: 78.6 ms
Wall time: 5.95 s


[{'timestamp': [0.0, 8.38],
  'text': " As big tech is getting hammered in today's selloff, CNBC's Magnificent 7 index dropping"},
 {'timestamp': [8.38, 14.78],
  'text': " more than 1%. Well, there's a new emerging threat to mega caps, massive spending in America's"},
 {'timestamp': [14.78, 19.44],
  'text': " dominance in the AI race. Deirdre Bosa digs into that for today's tech check. Hey, Dee."},
 {'timestamp': [20.18, 24.6],
  'text': " Hey, good morning, Leslie. So here's a name that our audience may want to write down,"},
 {'timestamp': [0.0, 6.68],
  'text': ' DeepSeq. This is a new free open source AI model that beats the latest open AI and meta models on'},
 {'timestamp': [6.68, 11.68],
  'text': ' key benchmarks. And it was made for a fraction of a fraction of the cost. Now, it was trained by a'},
 {'timestamp': [11.68, 17.44],
  'text': " Chinese research lab that used NVIDIA H800s. That's a lower performance version of the H100"},
 {'timestamp': [17.44, 22.88],
  'text': '

## Conclusion

HuggingFace's Serverless Inference API is unleasing AI super power with just an API call away. 
With ASR at your fingertip and the advancement of LLMs, the apps that we can create are only limited by our imagination!
Only question is what you will build with this?